# 40 — Developer · Debugger · Researcher

Three of the 47 specialists that ship with shipit_agent. Each is a fully-formed prompt + model + tool preset. Use them alone, or hand any of them to an Autopilot for long-running operation.

All three run against **Bedrock Llama** by default via `build_llm_from_env('bedrock')`.

In [ ]:
from pathlib import Path
import sys

# Resolve the repo root whether this cell is running from ./notebooks
# or from the repo root — mirrors the existing 01–36 notebook series.
ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_llm_from_env

# Bedrock Llama 4 Scout is the default model this library ships with
# (see `shipit_agent/config.py`). No model name required — the helper
# reads `AWS_REGION` / credentials from env and returns an adapter that
# works with every Autopilot / Agent class.
llm = build_llm_from_env("bedrock")
print("Bedrock LLM ready:", type(llm).__name__)


## 1. Load the specialist roster

`AgentRegistry` loads every `.json` definition shipped in `shipit_agent/agents/agents.json`, including the seven roles added by the specialist patch.

In [ ]:
from shipit_agent.agents import AgentRegistry

registry = AgentRegistry()
by_category = {}
for a in registry.all():
    by_category.setdefault(a.category, []).append(a.id)

for cat, ids in sorted(by_category.items()):
    print(f"{cat}:")
    for aid in ids:
        print(f"  - {aid}")


## 2. Generalist Developer — implement cleanly, verify, report

In [ ]:
from shipit_agent import Agent
from shipit_agent.builtins import get_builtin_tools

dev_def = registry.get("generalist-developer")
dev = Agent(
    llm=llm,
    prompt=dev_def.prompt,
    tools=get_builtin_tools(project_root="."),
    max_iterations=dev_def.maxIterations or 40,
    name=dev_def.name,
)
result = dev.run("Add a docstring to the function in shipit_agent/autopilot/budget.py named `BudgetPolicy.exceeded` explaining the return shape in one sentence. Do not change behaviour.")
print(result.output[:700])


## 3. Debugger — reproduce, isolate, explain, fix

The debugger specialist is hypothesis-driven: it refuses to write any fix until it has a minimal reproducer and a one-paragraph root-cause statement.

In [ ]:
dbg_def = registry.get("debugger")
dbg = Agent(
    llm=llm, prompt=dbg_def.prompt,
    tools=get_builtin_tools(project_root="."),
    max_iterations=dbg_def.maxIterations or 40,
    name=dbg_def.name,
)
result = dbg.run(
    "Investigate why `Autopilot.run()` might raise FileExistsError, describe the root cause, and explain the correct resolution flow."
)
print(result.output[:900])


## 4. Researcher + `research_brief` tool

The researcher pairs naturally with the new `research_brief` tool — web search + top-page skim + structured citations.

In [ ]:
from shipit_agent.tools.research_brief import ResearchBriefTool

researcher_def = registry.get("researcher")
researcher = Agent(
    llm=llm, prompt=researcher_def.prompt,
    tools=[ResearchBriefTool()],
    max_iterations=researcher_def.maxIterations or 20,
    name=researcher_def.name,
)
result = researcher.run("Brief me on recent Python dependency management tools (uv, poetry, pdm, rye). One-line tradeoff each, with citations.")
print(result.output[:900])


## 5. Researcher + Autopilot — long-form, budget-gated

Hand the researcher to Autopilot when the task needs more than one pass — e.g. multi-source synthesis with a deliverable.

In [ ]:
from shipit_agent.autopilot import Autopilot, BudgetPolicy
from shipit_agent.deep import Goal

goal = Goal(
    objective="Produce a two-page brief on the state of local LLM tooling (ollama, llama.cpp, mlx) in Q2 2026.",
    success_criteria=[
        "Covers all three ecosystems",
        "Cites at least three sources per ecosystem",
        "Ends with a recommendation table",
    ],
)
long_runner = Autopilot(
    llm=llm, goal=goal,
    tools=[ResearchBriefTool()],
    budget=BudgetPolicy(max_iterations=6, max_seconds=600, max_tool_calls=25),
)

result = long_runner.run(run_id="llm-tooling-brief")
print(f"status={result.status} · iters={result.iterations}")
print(result.output[:800])


## Summary

- 47 specialists preloaded; 7 added this release (developer, debugger, design-reviewer, PM, sales, CS, marketing-writer).
- Any specialist + Autopilot + a budget = production-ready long-running operation.

Next: **41 — design reviewer, product manager, sales, CS, marketing**.